# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a practical guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant-python) library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available **record sets**, **fields**, and their `@id`s.

> In mlcroissant, you can use `dataset.record_sets` to programmatically examine all record sets and their structure.

In [ ]:
from pprint import pprint

# Explore available record sets
print('Available Record Sets:')
for rs in dataset.record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}")
    print('  Fields:')
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print()
# Save main record set id (for use below)
main_record_set = dataset.record_sets[0].id if dataset.record_sets else None
print(f"First record set id to use: {main_record_set}")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Below, you can modify to explore any record set/field `@id` from the overview above.

In [ ]:
# Extract all data from all record sets into pandas DataFrames

record_sets = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records):
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Shape: {df.shape}")
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print('  No records found.')
# For continued analysis, pick a main record set (if available)
if len(dataframes) > 0:
    main_rs_id = record_sets[0]
    print(f"Selected main record set: {main_rs_id}")
else:
    main_rs_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

> The following EDA steps use typical numeric fields (such as `coverage`, `pept_count`, or `MW`) found in proteomics tables. _Replace with actual field `@id` as appropriate for your data structure._

In [ ]:
# Automatically select a numeric field for demonstration

import numpy as np

if main_rs_id is not None:
    df = dataframes[main_rs_id]

    # Try to automatically select a likely numeric field
    numeric_candidates = [col for col in df.columns if df[col].dtype in [np.int64, np.float64] or pd.api.types.is_numeric_dtype(df[col])]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
    else:
        # If dataset has no inferred numeric columns, try by prefix or fallback
        for cand in ["Coverage", "MW", "pept_count", "pi", "count", "abundance"]:
            if cand in df.columns:
                numeric_field = cand
                break
        else:
            numeric_field = None

    print(f"Using numeric field for filtering and normalization: {numeric_field}")

    # Set a threshold (use mean if possible)
    if numeric_field and pd.api.types.is_numeric_dtype(df[numeric_field]):
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 10

        # Filter records above threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize column
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()

        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by another field (for proteins, gene name or accession is typical)
        group_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            print(grouped_df.head())
    else:
        print("No suitable numeric field found for EDA.")
else:
    print("No suitable record set loaded for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we use `matplotlib` for histograms/scatterplots of quantitative variables.

In [ ]:
import matplotlib.pyplot as plt

if main_rs_id is not None and numeric_field in dataframes[main_rs_id]:
    plt.figure(figsize=(8, 5))
    df = dataframes[main_rs_id]
    df[numeric_field].hist(bins=30, color='skyblue', edgecolor='k')
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    # Optionally a scatter plot of two numeric columns
    num_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if len(num_candidates) > 1:
        plt.figure(figsize=(7, 5))
        plt.scatter(df[num_candidates[0]], df[num_candidates[1]], alpha=0.6, s=20)
        plt.xlabel(num_candidates[0])
        plt.ylabel(num_candidates[1])
        plt.title(f'Scatter: {num_candidates[0]} vs {num_candidates[1]}')
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we have:

- Loaded the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset's metadata and data using the `mlcroissant` library,
- Explored available record sets and fields by their `@id`,
- Loaded record set data into DataFrames and performed EDA, filtering by a numeric field and visualizing the results.

You can further extend this notebook by exploring additional record sets, visualizations, or downstream data analysis and modeling tasks relevant to proteomics and extracellular vesicle research.